# FastText Model Evaluation

Evaluate the FastText transaction classifier.

**Important**: FastText is a C++ library with Python bindings, not a PyTorch model.
It does not have a direct ONNX export path. Most optimization strategies (ONNX,
quantization, GPU EPs, OpenVINO) are **N/A** for FastText.

FastText is already extremely fast for text classification due to its shallow
architecture (bag-of-words with n-grams + linear classifier).

**Strategies evaluated**:
1. FastText baseline (CPU) -- native inference
2-8. N/A (documented below)

**Metrics**: model size, inference latency (p50/p95/p99), single-sample throughput, batch throughput

In [ ]:
import os
import sys
import time
import numpy as np
import pandas as pd
import fasttext

sys.path.insert(0, os.path.join(os.getcwd(), 'scripts'))
from benchmark_utils import (
    get_model_size_mb, benchmark_latency, benchmark_batch_throughput,
    print_benchmark_results, collect_result_row
)

MODEL_NAME = "fasttext"
MODEL_DIR = f"models/{MODEL_NAME}"

all_results = []

## Load sample data

In [ ]:
sample_texts = [
    "TESCO STORES 3456 -45.20 Friday",
    "DOMINOS #7548 -22.50 Thursday",
    "OLIVE GARDEN #9785 -67.30 Saturday",
    "AMAZON.COM -129.99 Monday",
    "SHELL OIL 12345 -55.00 Tuesday",
    "NETFLIX.COM -15.99 Wednesday",
    "WHOLE FOODS MKT -88.45 Sunday",
    "UBER *TRIP -18.75 Friday",
]

single_text = sample_texts[0]
batch_texts = sample_texts * 4  # 32 samples

print(f"Single sample: '{single_text}'")
print(f"Batch size: {len(batch_texts)}")

## 1. FastText Baseline (CPU)

In [ ]:
# Find the FastText model file (.bin or .ftz)
model_file = None
for candidate in ['model.bin', 'model.ftz', 'fasttext_model.bin', 'fasttext_model.ftz']:
    p = os.path.join(MODEL_DIR, candidate)
    if os.path.exists(p):
        model_file = p
        break

if model_file is None:
    # Search recursively
    for root, dirs, files in os.walk(MODEL_DIR):
        for f in files:
            if f.endswith(('.bin', '.ftz')):
                model_file = os.path.join(root, f)
                break

if model_file is None:
    print(f"FastText model file not found in {MODEL_DIR}. Contents:")
    for root, dirs, files in os.walk(MODEL_DIR):
        for f in files:
            print(f"  {os.path.join(root, f)}")
    raise FileNotFoundError(f"Cannot find FastText model in {MODEL_DIR}")

ft_model = fasttext.load_model(model_file)
model_size = get_model_size_mb(model_file)
print(f"Loaded FastText model from {model_file}")
print(f"Model size: {model_size:.2f} MB")

In [ ]:
def predict_fasttext_single(text):
    labels, probs = ft_model.predict(text)
    return labels[0], probs[0]

def predict_fasttext_batch(texts):
    results = [ft_model.predict(t) for t in texts]
    return results

latency_results = benchmark_latency(predict_fasttext_single, single_text)
batch_results = benchmark_batch_throughput(predict_fasttext_batch, batch_texts)
results_baseline = {**latency_results, **batch_results}

print_benchmark_results(results_baseline, MODEL_NAME, "native_cpu", model_size)
all_results.append(collect_result_row(MODEL_NAME, "native_cpu", "CPU", model_size, results_baseline))

## 2-8. Optimization Strategies: N/A

The following strategies are **not applicable** to FastText:

| Strategy | Status | Reason |
|---|---|---|
| ONNX (no opt) | N/A | FastText uses a custom C++ backend; no direct ONNX export path exists |
| ONNX + graph opt | N/A | Depends on ONNX export |
| Dynamic quantization | N/A | Depends on ONNX export |
| Static quantization (aggressive) | N/A | Depends on ONNX export |
| Static quantization (conservative) | N/A | Depends on ONNX export |
| GPU EPs (CUDA/TensorRT) | N/A | FastText is CPU-only by design; its shallow architecture is already very fast on CPU |
| OpenVINO EP | N/A | Depends on ONNX export |

**Why FastText is still a strong candidate**: Despite lacking optimization paths, FastText
is typically orders of magnitude faster than transformer models at inference time due to
its simple bag-of-words architecture. Its baseline performance may already outperform
optimized transformer variants.

In [ ]:
na_strategies = [
    "onnx_no_opt", "onnx_optimized", "dynamic_quant",
    "static_quant_aggressive", "static_quant_conservative",
    "onnx_cuda", "onnx_tensorrt", "onnx_openvino"
]

for strategy in na_strategies:
    all_results.append({
        "option": f"{MODEL_NAME}_{strategy}",
        "model": MODEL_NAME,
        "config": strategy,
        "hardware": "N/A",
        "model_size_mb": None,
        "p50_latency_ms": None,
        "p95_latency_ms": None,
        "p99_latency_ms": None,
        "throughput_fps": None,
        "batch_throughput_fps": None,
    })

print("N/A entries added for non-applicable strategies.")

## Summary

In [ ]:
results_df = pd.DataFrame(all_results)
print(results_df.to_string(index=False))
results_df.to_csv(f"results/{MODEL_NAME}_evaluation.csv", index=False)
print(f"\nResults saved to results/{MODEL_NAME}_evaluation.csv")